# Load Packages

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from os.path import join
from tqdm.auto import tqdm
import joblib
import torch
sys.path.append("../../")

from src.configs.default_configs import fp_checkpoint_folder, fp_checkpoint_folder, fn_ue_perf, fn_consolidated
from src.configs.crc_config import data_name
from src.evaluation.consolidate import consolidate_pred_perf, consolidate_ue_perf, consolidate_ood_perf
from src.evaluation.perf_eval import display_pred_perf
from src.evaluation.ue_metrics import display_ue_perf, display_ood_perf
from src.display.display_df import df_to_latex

start_seed, num_seeds = 2024, 10
seed_list=list(range(start_seed, start_seed+num_seeds))
fp_evaluation = join(fp_checkpoint_folder, fn_ue_perf, data_name)
fp_consolidated = join(fp_checkpoint_folder, fn_consolidated, data_name)

# Prediction Perf

In [ ]:
pred_perf_df = consolidate_pred_perf(seed_list, fp_evaluation, sp=3)
display_pred_perf(pred_perf_df[["AUC", "Accuracy"]], consolidated=True)
pred_perf_df.to_csv(join(fp_consolidated, "pred_perf.csv"))

In [ ]:
print(df_to_latex(pred_perf_df[["AUC", "Accuracy"]], column_format_dict={"Accuracy": "max", "AUC": "max"}))

# UE Evaluation

In [ ]:
ue_perf_df = consolidate_ue_perf(seed_list, fp_evaluation,) #exclude_columns="Pval")
display_ue_perf(ue_perf_df, consolidated=True)
ue_perf_df.to_csv(join(fp_consolidated, "ue_perf.csv"))

In [ ]:
high_cols = ["Pearson's Correlation"]
low_cols = ["AURC (0/1 Loss)","Sigma-Risk Score (0.1)","Sigma-Risk Score (0.2)","Sigma-Risk Score (0.3)","Sigma-Risk Score (0.4)"]
column_format_dict = {col:"max" for col in high_cols}
column_format_dict.update({col:"min" for col in low_cols})
print(df_to_latex(ue_perf_df, column_format_dict=column_format_dict))